# NullFusion — Null-Space Conditional Fusion (proposal7)

**Goal:** beat SOTA on Chikusei x4 with improved capacity (width=128, depth=12)

Architecture: `X_hat = pinv(yH, yM) + P_N( f_theta(conditioning) )`

The observation-consistent component is solved in closed form (exact `A(X_hat)==[yH;yM]` is an algebraic identity, ~1e-5), and the network only ever fills the null space the sensors provably cannot see — so it cannot hallucinate the observable part (P1 admissible ambiguity).

### Target SOTA: FeINFN 52.47 dB on CAVE x4 Nikon D700 SRF; adapting to Chikusei x4 protocol

In [ ]:
import subprocess, glob, os, shutil

# Recursively find .whl files under /kaggle/input/
whls = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.whl'):
            whls.append(os.path.join(root, f))
whls.sort()
print(f'Found wheels: {whls}')
if not whls:
    raise RuntimeError('No torch wheels found')
for w in whls:
    base = os.path.basename(w)
    fixed = base.replace('cu121-cp312', '+cu121-cp312')
    dst = os.path.join('/tmp', fixed)
    shutil.copy2(w, dst)
    print(f'Installing {dst}')
    subprocess.check_call(['pip', 'install', '--force-reinstall', '--no-deps', dst])
print('torch install done')

In [ ]:
import torch, sys
print('torch', torch.__version__)
assert torch.cuda.is_available(), 'CUDA not available'
p = torch.cuda.get_device_properties(0)
print(f'gpu {p.name} {p.total_memory/2**30:.1f}GB sm_{p.major}{p.minor}')

In [ ]:
!pip install -q scipy scikit-image matplotlib einops

In [ ]:
import os, sys, glob

# Find the repo root
REPO = '/kaggle/working/repo'
if not os.path.exists(REPO):
    import zipfile
    zip_path = '/kaggle/input/hyperspectral-image-fusion/repo.zip'
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(REPO)
        print(f'Extracted repo to {REPO}')
    else:
        raise RuntimeError('Repo zip not found')

sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'common'))
print('REPO =', REPO)

from nullfusion.nullfusion import NullFusionNet, NullFusionConfig
print('NullFusionNet import OK')

cfg = NullFusionConfig(
    width=128,       # increased from 96 for SOTA capacity
    prior_depth=12,  # increased from 8
    enc_depth=6,     # increased from 4
    cross_attn_heads=8,
    use_attn=True,
    rank=31          # keep full spectral capacity (bands=31)
)
print(f'Config: width={cfg.width}, prior_depth={cfg.prior_depth}, enc_depth={cfg.enc_depth}')

## 1. Chikusei Dataset Discovery

In [ ]:
from hsifusion.io_utils import discover_dataset, available_splits, find_pairs, list_hsi

spec_root = discover_dataset(["Chikusei"], required=True)
splits = available_splits(spec_root)
print('Chikusei splits:', splits)
train_pairs = find_pairs(spec_root, splits.get('Train', 'Train'))
test_scenes = list_hsi(spec_root, splits.get('Test', 'Test'))
print(f'train pairs = {len(train_pairs)}   test scenes = {len(test_scenes)}')

## 2. Set Nikon D700 SRF (the SOTA protocol SRF)
The Nikon D700 SRF is the standard spectral response function used in the unified protocol.
Without it, the problem is easier (Gaussian SRF cond=1.42 vs Nikon cond=1.86).

In [ ]:
import torch
from hsifusion.config import Config

# Load the Nikon D700 SRF - this is the protocol that matches FeINFN/BDT SOTA
cfg_protocol = Config.paper_core()
srf_path = cfg_protocol.source_root / 'srfs' / 'nikon_d700.npy'
if srf_path.exists():
    srf = torch.from_numpy(np.load(srf_path)).float()
    print(f'Loaded SRF from: {srf_path}')
    print(f'SRF shape: {srf.shape}')
    
    # Apply SRF to the nullfusion model
    from nullfusion.nullfusion.model import NullFusionNet
    from nullfusion.nullfusion.model import NullFusionConfig
    
    model = NullFusionNet(NullFusionConfig())
    model.set_srf(srf)
    print('SRF applied to model')
else:
    print(f'SRF not found at: {srf_path}')
    print('Looking for SRF in common/hsifusion/srfs/')
    srf_dir = Path('/kaggle/working/common/hsifusion/srfs')
    if srf_dir.exists():
        print(f'SRF files in /kaggle/working/common/hsifusion/srfs/:')
        for f in srf_dir.glob('*.npy'):
            print(f'  - {f.name}')

## 3. Train NullFusionNet on Chikusei x4

### Training Configuration (based on SOTA push protocol):
- **Degradation**: Wald Gaussian σ=1.2, 9×9 kernel, ×4 decimation
- **SRF**: Nikon D700 (standard SOTA protocol)
- **Data range**: constant 1.0 (never per-image max)
- **Evaluation**: Hann-weighted overlapping tiles, no centre crops
- **Epochs**: time-budgeted (9h on T4x2 → ~2000 epochs achievable)
- **Optimizer**: AdamW, cosine LR decay
- **Gradient accumulation**: ×2

### Why this capacity should beat SOTA:
- width=128 → ~5M parameters (vs 2.75M baseline)
- prior_depth=12 with cross-attn + spatial self-attn + spectral mixing
- Full rank=31 bottleneck (no artificial constraint)
- Longer training: FeINFN reproduction at 1095 epochs was 'still rising ~0.01 dB/20 ep'
- Target: 2000+ epochs should reach/beat 52.47 dB on Chikusei

In [ ]:
from nullfusion.nullfusion.train_sota import TrainConfig, train

cfg = TrainConfig(
    epochs=2000,              # time-budgeted for T4x2 (9h)
    batch_size=16,
    lr=1e-4,                 # AdamW base LR
    lr_decay='cosine',       # cosine decay
    grad_accum=2,            # gradient accumulation ×2
    save_every=200,
    eval_every=50,
    use_amp=True,            # automatic mixed precision for T4
    clip_grad=1.0
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

# Start training
train(cfg, device)


## 4. Results & Evaluation

In [ ]:
import json, os

# Load best results
results_path = 'best_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        res = json.load(f)
    print('='*60)
    print('CHIKUSEI X4 RESULTS')
    print('='*60)
    print(f'Protocol: {res.get("protocol", "N/A")}')
    print(f'Mean PSNR: {res.get("mean_psnr", "N/A"):.2f} dB')
    print(f'Mean SSIM: {res.get("mean_ssim", "N/A"):.4f}')
    print(f'Mean SAM:  {res.get("mean_sam", "N/A"):.3f}°')
    print(f'Mean ERGAS: {res.get("mean_ergas", "N/A"):.3f}')
    print(f'Params: {res.get("nparams", "N/A")/1e6:.1f}M')
    print(f'Best epoch: {res.get("best_epoch", "N/A")}')
    print('='*60)

# Compare to SOTA
sota_psnr = 52.47  # FeINFN on CAVE x4 Nikon
achieved = res.get('mean_psnr', 0)
gap = sota_psnr - achieved
if achieved > 52.47:
    print(f'🏆 NEW SOTA: {achieved:.2f} dB (beats {sota_psnr} dB by {gap:.2f} dB!)')
elif achieved >= 50.31:
    print(f'📈 Improved over NullFusion v4 (50.31 dB) by {gap:.2f} dB')
else:
    print(f'Current: {achieved:.2f} dB, gap to SOTA: {gap:.2f} dB')
    print('Train longer (2000+ epochs) or increase width for better results')